# Databricks/PySpark for Data Analysts/Engineers — Module 7: Window Functions in Spark

Window functions let you compute values "in the context" of a group of rows (e.g. a
ranking within a city, a running total over time), WITHOUT collapsing the result to one
row per group -- unlike `groupBy`. This is exactly the same concept as in Module 11 of
the SQL course, now via `pyspark.sql.window.Window`.

## Table of contents
1. [Window: partitionBy and orderBy](#sec1)
2. [row_number, rank, dense_rank](#sec2)
3. [Running totals](#sec3)
4. [lag and lead](#sec4)
5. [Summary and exercises](#sec5)


We start (or grab the already-running) `SparkSession` -- the entry point to everything in Spark. In Databricks, `spark` is already available automatically in every notebook; the same code also works locally.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("spark_course").getOrCreate()
spark


Let's load our shared dataset -- online store data (`customers`, `products`, `employees`, `orders`, `order_items`), the exact same schema as the SQL course, but this time as Spark DataFrames.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)

# --- customers -----------------------------------------------------------
n_customers = 300
cities = ["Warsaw", "Krakow", "Gdansk", "Wroclaw", "Poznan"]
segments = ["New", "Standard", "Premium"]
segment_probs = [0.35, 0.45, 0.20]

customers_pd = pd.DataFrame({
    "customer_id": np.arange(1, n_customers + 1),
    "city": rng.choice(cities, size=n_customers, p=[0.30, 0.25, 0.15, 0.15, 0.15]),
    "segment": rng.choice(segments, size=n_customers, p=segment_probs),
    "days_since_signup": rng.integers(1, 900, size=n_customers),
})

# --- products ------------------------------------------------------------
n_products = 40
categories = ["Electronics", "Clothing", "Home", "Sports", "Books"]
avg_prices = {"Electronics": 1200, "Clothing": 150, "Home": 250, "Sports": 300, "Books": 50}

product_categories = rng.choice(categories, size=n_products)
products_pd = pd.DataFrame({
    "product_id": np.arange(1, n_products + 1),
    "category": product_categories,
    "price": [
        round(max(5.0, rng.normal(avg_prices[c], avg_prices[c] * 0.3)), 2)
        for c in product_categories
    ],
})

# --- employees ----------------------------------------------------------
n_employees = 12
regions = ["North", "South", "East", "West"]
employees_pd = pd.DataFrame({
    "employee_id": np.arange(1, n_employees + 1),
    "region": rng.choice(regions, size=n_employees),
    "department": rng.choice(["Sales", "Support"], size=n_employees, p=[0.6, 0.4]),
})

# --- orders ------------------------------------------------------------
n_orders = 3000
order_customer_id = rng.choice(customers_pd["customer_id"], size=n_orders)
order_employee_id = rng.choice(employees_pd["employee_id"], size=n_orders)

segment_map = customers_pd.set_index("customer_id")["segment"]
order_segment = pd.Series(order_customer_id).map(segment_map).values

# cancellation probability depends on customer segment (deliberately built in)
cancel_prob = np.where(
    order_segment == "New", 0.22,
    np.where(order_segment == "Standard", 0.10, 0.04),
)
cancelled = rng.binomial(1, cancel_prob)
order_status = np.where(
    cancelled == 1, "Cancelled",
    rng.choice(["Placed", "Shipped", "Delivered"], size=n_orders, p=[0.2, 0.3, 0.5]),
)

base_date = pd.Timestamp("2024-01-01")
day_offset = rng.integers(0, 730, size=n_orders)

orders_pd = pd.DataFrame({
    "order_id": np.arange(1, n_orders + 1),
    "customer_id": order_customer_id,
    "employee_id": order_employee_id,
    "order_date": [base_date + pd.Timedelta(days=int(d)) for d in day_offset],
    "status": order_status,
})

# --- order_items ------------------------------------------------------
item_rows = []
item_id = 1
product_price = products_pd.set_index("product_id")["price"]

for order_id in orders_pd["order_id"]:
    n_items = rng.integers(1, 5)
    chosen_products = rng.choice(products_pd["product_id"], size=n_items, replace=False)
    for product_id in chosen_products:
        quantity = int(rng.integers(1, 4))
        discount = rng.choice([0.0, 0.05, 0.10, 0.15], p=[0.55, 0.20, 0.15, 0.10])
        item_rows.append({
            "item_id": item_id,
            "order_id": int(order_id),
            "product_id": int(product_id),
            "quantity": quantity,
            "unit_price": float(product_price[product_id]),
            "discount": float(discount),
        })
        item_id += 1

order_items_pd = pd.DataFrame(item_rows)

# --- convert to Spark DataFrames ---------------------------------------
customers = spark.createDataFrame(customers_pd)
products = spark.createDataFrame(products_pd)
employees = spark.createDataFrame(employees_pd)
orders = spark.createDataFrame(orders_pd)
order_items = spark.createDataFrame(order_items_pd)

customers.show(5)


<a id="sec1"></a>
## 1. Window: partitionBy and orderBy

A `Window` object defines a "window" -- how to split data into groups (`partitionBy`) and
in what order to scan them within each group (`orderBy`). It computes nothing by itself
-- we pass it to a window function via `.over(window)`.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

window_by_city = Window.partitionBy("city").orderBy(F.col("days_since_signup").desc())

customers.withColumn(
    "position_in_city", F.row_number().over(window_by_city)
).orderBy("city", "position_in_city").show(10)


> ⚡ **Window.partitionBy() is not the same as groupBy()**
>
> `groupBy()` COLLAPSES multiple rows into one per group. `Window.partitionBy()` collapses NOTHING -- every original row stays in place, it just gets an extra column computed 'in the context' of its group. That's the main reason window functions are so useful: ranking, running totals, or comparing with the previous row -- all without losing the detail data.

<a id="sec2"></a>
## 2. row_number, rank, dense_rank

The three most common ranking functions, differing in how they handle TIES (identical
values in the `orderBy` column):

- `row_number()` -- always a unique number 1, 2, 3, ... (even on a tie),
- `rank()` -- tied rows get the same number, and the next number SKIPS ahead (1, 1, 3),
- `dense_rank()` -- tied rows get the same number, and the next number does NOT skip (1, 1, 2).

In [ ]:
price_window = Window.partitionBy("category").orderBy(F.col("price").desc())

products.withColumn("row_number", F.row_number().over(price_window)) \
    .withColumn("rank", F.rank().over(price_window)) \
    .withColumn("dense_rank", F.dense_rank().over(price_window)) \
    .orderBy("category", "row_number").show(15)


> ⚠️ **Top-N per group -- the classic row_number() pattern**
>
> The most common use case: 'find the 3 most expensive products in each category'. You compute `row_number()` over a window sorted descending by price, then filter `WHERE row_number <= 3` -- something plain `groupBy()` cannot express.

In [ ]:
products.withColumn("ranking", F.row_number().over(price_window)).filter(
    F.col("ranking") <= 3
).orderBy("category", "ranking").show(20)


<a id="sec3"></a>
## 3. Running totals

By giving the window a row range through `.rowsBetween(...)`, we can compute running
sums/averages -- e.g. `Window.unboundedPreceding` to the current row means "every row
from the start of the partition up to now".

In [ ]:
running_window = (
    Window.partitionBy("customer_id")
    .orderBy("order_date")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

orders_with_value = orders.join(order_items, "order_id").groupBy(
    "order_id", "customer_id", "order_date"
).agg(F.sum(F.col("quantity") * F.col("unit_price") * (1 - F.col("discount"))).alias("value"))

orders_with_value.withColumn(
    "running_total", F.round(F.sum("value").over(running_window), 2)
).orderBy("customer_id", "order_date").show(10)


> 🧱 **The default window range when you skip rowsBetween**
>
> If you only give `orderBy` without `.rowsBetween(...)`, Spark computes a running total by default EXACTLY as above (`unboundedPreceding` to `currentRow`) -- but spelling out the range explicitly is more readable and avoids surprises with ties in the `orderBy` column (rows with the same value can then 'stick together' into one range group, the so-called `RANGE` mode instead of `ROWS`).

<a id="sec4"></a>
## 4. lag and lead

`lag(column, n)` returns the value from `n` POSITIONS EARLIER in the window, `lead(column, n)`
-- `n` positions LATER. A classic use case: comparing with the same customer's previous
or next order.

In [ ]:
time_window = Window.partitionBy("customer_id").orderBy("order_date")

orders_with_value.withColumn(
    "previous_value", F.lag("value", 1).over(time_window)
).withColumn(
    "change", F.round(F.col("value") - F.col("previous_value"), 2)
).orderBy("customer_id", "order_date").show(10)


> ⚠️ **The first/last row in a partition -- lag/lead returns NULL**
>
> For a customer's earliest order, `lag(...)` has no previous value to pull -- it returns `NULL`. That's expected behavior, but you need to handle it (e.g. `fillna`, or deliberately leave it as missing data), especially before further calculations on the `change` column.

<a id="sec5"></a>
## 5. Summary and exercises

In this module we covered:
- `Window.partitionBy().orderBy()` -- defining a window without collapsing rows (unlike `groupBy`),
- `row_number()`, `rank()`, `dense_rank()` -- the differences in handling ties,
- the Top-N-per-group pattern via `row_number()` + a filter,
- running sums/averages via `.rowsBetween(...)`,
- `lag()`/`lead()` -- accessing values from neighboring rows in a window.

### 📝 Exercise 1: Ranking employees within a region

For `employees`, compute `row_number()` over a window partitioned by `region` and ordered by `employee_id`.

<details>
<summary>Show solution</summary>

```python
from pyspark.sql.window import Window
from pyspark.sql import functions as F

window = Window.partitionBy("region").orderBy("employee_id")
employees.withColumn("position_in_region", F.row_number().over(window)).orderBy(
    "region", "position_in_region"
).show()
```

</details>

### 📝 Exercise 2: The 2 cheapest products in each category

Use `row_number()` over a window sorted ASCENDING by `price` within `category`, to find the 2 cheapest products in each category.

<details>
<summary>Show solution</summary>

```python
window = Window.partitionBy("category").orderBy(F.col("price").asc())
products.withColumn("ranking", F.row_number().over(window)).filter(
    F.col("ranking") <= 2
).orderBy("category", "ranking").show()
```

</details>

### 📝 Exercise 3: A running total of order count over time

Compute the order count per day (`order_date`), then a running total of that count over time (a single partition -- the whole thing, ordered by date).

<details>
<summary>Show solution</summary>

```python
daily = orders.groupBy("order_date").agg(F.count("*").alias("count"))

window = Window.orderBy("order_date").rowsBetween(
    Window.unboundedPreceding, Window.currentRow
)
daily.withColumn("running_total", F.sum("count").over(window)).orderBy(
    "order_date"
).show(10)
```

</details>

### 📝 Exercise 4: Comparing with the previous order

For each customer, use `lag()` to add a column with the STATUS of their previous order (ordered by `order_date`).

<details>
<summary>Show solution</summary>

```python
window = Window.partitionBy("customer_id").orderBy("order_date")
orders.withColumn(
    "previous_status", F.lag("status", 1).over(window)
).orderBy("customer_id", "order_date").show(10)
```

</details>

> 🔥 **Challenge: customers with an increasing value across their last three orders**
>
> For each customer, compute the value of every order (as in section 3), add columns `previous_value` and `value_two_back` (`lag(1)` and `lag(2)`), then find customers whose last THREE orders have a STRICTLY increasing value (`value_two_back < previous_value < value`).

<details>
<summary>Show solution</summary>

```python
window = Window.partitionBy("customer_id").orderBy("order_date")

with_history = orders_with_value.withColumn(
    "previous_value", F.lag("value", 1).over(window)
).withColumn(
    "value_two_back", F.lag("value", 2).over(window)
)

increasing = with_history.filter(
    (F.col("value_two_back") < F.col("previous_value"))
    & (F.col("previous_value") < F.col("value"))
)
increasing.select(
    "customer_id", "order_date", "value_two_back",
    "previous_value", "value"
).orderBy("customer_id", "order_date").show(20)
```

Note that this exercise depends on the result from section 3 (orders_with_value) -- if you're running this exercise on its own, first compute each order's value as shown in section 3.

</details>

## What's next?

In **Module 8** we'll cover UDFs (user-defined functions) -- when they're needed, their
performance cost compared to built-in functions, and what pandas UDFs are.